In [2]:
import os

# SET THIS TO YOUR PROJECT ROOT PATH
os.chdir(r"C:\JupyterProjects\Stock_ML_Project")

print("Current working directory:", os.getcwd())

Current working directory: C:\JupyterProjects\Stock_ML_Project


In [3]:
# 05_regression_baseline_all_stocks.ipynb
# Paste this entire cell into a Jupyter notebook and run.

# -----------------------
# Imports & helpers
# -----------------------
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make displays nicer in notebook
pd.options.display.float_format = '{:,.6f}'.format

# -----------------------
# Config / file list
# -----------------------
PROJECT_ROOT = Path('.')  # adjust if your notebook runs from a subfolder
processed_dir = PROJECT_ROOT / "data" / "processed"
figures_reg_dir = PROJECT_ROOT / "figures" / "models" / "regression"
results_dir = PROJECT_ROOT / "results"

# Ensure output directories exist
for p in [figures_reg_dir, results_dir]:
    p.mkdir(parents=True, exist_ok=True)

# List of model-ready files (change names if your filenames are different)
files = {
    "RELIANCE": processed_dir / "reliance_model_ready.csv",
    "TCS": processed_dir / "tcs_model_ready.csv",
    "HDFCBANK": processed_dir / "hdfcbank_model_ready.csv"
}

# Models to evaluate
models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "SVR_rbf": SVR(kernel='rbf', C=100, gamma='scale')
}

# -----------------------
# Utility functions
# -----------------------
def evaluate_regression(model, X_train, y_train, X_test, y_test, scale_for_svr=False):
    """
    Fit model, compute preds and return metrics + preds.
    If scale_for_svr=True, expects X_train, X_test unscaled and will scale internally.
    """
    # For SVR only: scale features (keeps other models unscaled)
    if isinstance(model, SVR) and scale_for_svr:
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return {"mae": mae, "rmse": rmse, "r2": r2, "preds": preds}

def safe_find(col_list, substr):
    """Return first column name in col_list containing substr, or None."""
    for c in col_list:
        if substr in c:
            return c
    return None

# -----------------------
# Main loop over stocks
# -----------------------
all_results = []  # will hold rows for results CSV

for ticker, filepath in files.items():
    print(f"\n=== Processing {ticker} -> {filepath} ===")
    if not filepath.exists():
        print("  WARNING: file not found, skipping:", filepath)
        continue

    df = pd.read_csv(filepath)
    print("  Loaded shape:", df.shape)

    # auto-detect columns
    # Regression target: prefer 'Target_Reg' or containing 'Target_Reg' or 'Next_Close'
    target_reg_col = safe_find(df.columns, "Target_Reg") or safe_find(df.columns, "Next_Close") or safe_find(df.columns, "Next_Close_".upper())
    # classification (not used here) but detect anyway:
    target_cls_col = safe_find(df.columns, "Target_Cls") or safe_find(df.columns, "Target")
    split_col = safe_find(df.columns, "Split") or safe_find(df.columns, "split")

    if target_reg_col is None or split_col is None:
        print("  ERROR: Could not detect required target or split columns in", filepath)
        print("  Columns present:", list(df.columns))
        continue

    print("  Using regression target:", target_reg_col, ", split column:", split_col)

    # Prepare X, y
    drop_cols = [target_reg_col, target_cls_col, split_col] if target_cls_col else [target_reg_col, split_col]
    drop_cols = [c for c in drop_cols if c in df.columns]  # keep only existing
    X = df.drop(columns=drop_cols)
    y = df[target_reg_col]

    # Chronological split using the 'Split' column values 'Train' / 'Test' (case-insensitive)
    train_mask = df[split_col].astype(str).str.lower() == "train"
    test_mask  = df[split_col].astype(str).str.lower() == "test"

    X_train = X[train_mask].reset_index(drop=True)
    X_test  = X[test_mask].reset_index(drop=True)
    y_train = y[train_mask].reset_index(drop=True)
    y_test  = y[test_mask].reset_index(drop=True)

    print(f"  Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
    if X_train.shape[0] == 0 or X_test.shape[0] == 0:
        print("  ERROR: empty train or test set, skipping")
        continue

    # Some models (SVR) require scaling; we will scale only when calling evaluate_regression
    for model_name, model_obj in models.items():
        # For reproducibility, re-instantiate model when necessary (some sklearn models retain state)
        # Use the class and its params to recreate a fresh instance
        ModelClass = model_obj.__class__
        # copy params
        try:
            params = model_obj.get_params()
            cur_model = ModelClass(**params)
        except Exception:
            cur_model = ModelClass()

        # Scale only for SVR
        scale_for_svr = model_name.lower().startswith("svr")
        result = evaluate_regression(cur_model, X_train, y_train, X_test, y_test, scale_for_svr=scale_for_svr)

        # Save results row
        all_results.append({
            "Ticker": ticker,
            "Model": model_name,
            "MAE": result["mae"],
            "RMSE": result["rmse"],
            "R2": result["r2"],
            "TrainRows": X_train.shape[0],
            "TestRows": X_test.shape[0]
        })

        # Save predictions (for plotting)
        preds = result["preds"]
        preds_df = pd.DataFrame({
            "Actual": y_test.values,
            "Predicted": preds
        })
        preds_csv = results_dir / f"{ticker}_{model_name}_preds.csv"
        preds_df.to_csv(preds_csv, index=False)

    # After evaluating all models for this ticker, create a single comparison plot (Actual vs best Pred)
    # Choose 'best' by RMSE from the models evaluated
    ticker_rows = [r for r in all_results if r["Ticker"] == ticker]
    # find model with min RMSE
    best_row = min(ticker_rows, key=lambda x: x["RMSE"])
    best_model_name = best_row["Model"]
    print(f"  Best model by RMSE for {ticker}: {best_model_name} (RMSE={best_row['RMSE']:.4f})")

    # load preds for that best model
    best_preds_df = pd.read_csv(results_dir / f"{ticker}_{best_model_name}_preds.csv")

    # Plot actual vs predicted (time series)
    plt.figure(figsize=(12,5))
    plt.plot(best_preds_df["Actual"].values, label="Actual", linewidth=2)
    plt.plot(best_preds_df["Predicted"].values, label=f"Predicted ({best_model_name})", alpha=0.9)
    plt.title(f"{ticker} — Actual vs Predicted ({best_model_name})")
    plt.xlabel("Test sample index (chronological)")
    plt.ylabel("Next Day Close (target)")
    plt.legend()
    plt.tight_layout()

    plot_path = figures_reg_dir / f"{ticker}_best_pred_vs_actual.png"
    plt.savefig(plot_path, dpi=200)
    plt.close()
    print("  Saved plot to:", plot_path)

# -----------------------
# Save combined results
# -----------------------
results_df = pd.DataFrame(all_results)
results_csv = results_dir / "regression_results.csv"
results_df.to_csv(results_csv, index=False)
print("\nCompleted. Summary saved to:", results_csv)
display(results_df.sort_values(["Ticker","Model"]))



=== Processing RELIANCE -> data\processed\reliance_model_ready.csv ===
  Loaded shape: (1460, 9)
  Using regression target: Target_Reg , split column: Split
  Train size: 1168, Test size: 292
  Best model by RMSE for RELIANCE: LinearRegression (RMSE=24.4245)
  Saved plot to: figures\models\regression\RELIANCE_best_pred_vs_actual.png

=== Processing TCS -> data\processed\tcs_model_ready.csv ===
  Loaded shape: (1460, 9)
  Using regression target: Target_Reg , split column: Split
  Train size: 1168, Test size: 292
  Best model by RMSE for TCS: LinearRegression (RMSE=73.0222)
  Saved plot to: figures\models\regression\TCS_best_pred_vs_actual.png

=== Processing HDFCBANK -> data\processed\hdfcbank_model_ready.csv ===
  Loaded shape: (1460, 9)
  Using regression target: Target_Reg , split column: Split
  Train size: 1168, Test size: 292
  Best model by RMSE for HDFCBANK: LinearRegression (RMSE=14.3212)
  Saved plot to: figures\models\regression\HDFCBANK_best_pred_vs_actual.png

Completed. 

,Ticker,Model,MAE,RMSE,R2,TrainRows,TestRows
9,HDFCBANK,DecisionTree,27.965709,38.467151,0.632178,1168,292
8,HDFCBANK,LinearRegression,10.573950,14.321181,0.949018,1168,292
10,HDFCBANK,RandomForest,23.768494,35.222402,0.691613,1168,292
11,HDFCBANK,SVR_rbf,32.810919,52.579410,0.312790,1168,292
1,RELIANCE,DecisionTree,142.588771,168.794178,-1.106861,1168,292
0,RELIANCE,LinearRegression,18.576166,24.424498,0.955886,1168,292
2,RELIANCE,RandomForest,142.676245,171.093790,-1.164659,1168,292
3,RELIANCE,SVR_rbf,211.172994,259.213965,-3.968642,1168,292
5,TCS,DecisionTree,441.700867,519.044507,-1.892308,1168,292
4,TCS,LinearRegression,56.791238,73.022208,0.942754,1168,292
